In [1]:
import csv
from dataclasses import dataclass
from typing import List

# --- 1. Data Schema ---
@dataclass
class Category:
    label: str
    words: List[str]
    difficulty: int

@dataclass
class ConnectionsPuzzle:
    puzzle_id: int
    categories: List[Category]

    @property
    def raw_board(self) -> List[str]:
        return [word.lower() for cat in self.categories for word in cat.words]

# --- 2. Ingestion Pipeline ---
def load_puzzles_from_csv(filepath: str) -> List[ConnectionsPuzzle]:
    # Map the text difficulty to the integer gradient for your solver
    difficulty_map = {
        "easy": 1,       # Yellow
        "medium": 2,     # Green
        "hard": 3,       # Blue
        "very hard": 4   # Purple
    }
    
    puzzles = []
    
    with open(filepath, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        # Skip the header row if you have one
        next(reader, None) 
        
        all_rows = list(reader)
        
        # Group every 4 rows into a single puzzle
        puzzle_counter = 1
        for i in range(0, len(all_rows), 4):
            game_rows = all_rows[i:i+4]
            
            # Safety check: Ensure we didn't hit a partial game at the end of the file
            if len(game_rows) != 4:
                print(f"Warning: Incomplete puzzle found at row {i}. Skipping.")
                break
                
            categories = []
            for row in game_rows:
                # Extract your columns based on your layout
                words = [row[0].strip(), row[1].strip(), row[2].strip(), row[3].strip()]
                label = row[4].strip()
                difficulty_str = row[5].strip().lower()
                
                # Convert the difficulty string to your algorithmic weight
                diff_int = difficulty_map.get(difficulty_str, 0)
                
                categories.append(Category(label=label, words=words, difficulty=diff_int))
            
            # Construct the full board state
            puzzle = ConnectionsPuzzle(puzzle_id=puzzle_counter, categories=categories)
            puzzles.append(puzzle)
            puzzle_counter += 1
            
    return puzzles

# --- 3. Execution ---
if __name__ == "__main__":
    # Example usage:
    parsed_games = load_puzzles_from_csv('output.csv')
    print(f"Successfully loaded {len(parsed_games)} puzzles.")
    print(parsed_games[0].raw_board)

Successfully loaded 915 puzzles.
['curses', 'fudge', 'blast', 'crud', 'choral', 'jazz', 'rap', 'americana', 'lord', 'please', 'sheesh', 'brother', 'heavens', 'gracious', 'mercy', 'dear']


In [ ]:
import numpy as np
from typing import List, Dict

class AffinityEngine:
    def __init__(self, embeddings: Dict[str, np.ndarray], alpha: float = 0.0):
        self.embeddings = embeddings
        self.vector_size = len(next(iter(embeddings.values()))) # e.g., 300
        self.alpha = alpha

    def get_word_vector(self, word: str) -> np.ndarray:
        """Fetches the vector, with a zero-vector fallback for OOV words."""
        # NYT sometimes uses hyphenated or compound words, so safe fallback is needed
        return self.embeddings.get(word, np.zeros(self.vector_size))

    def extract_semantic_matrix(self, words: List[str]) -> np.ndarray:
        """
        Calculates the 16x16 cosine similarity matrix for the given words.
        Cosine Similarity = (A dot B) / (||A|| * ||B||)
        """
        # 1. Stack the 16 vectors into a single matrix of shape (16, 300)
        vector_matrix = np.array([self.get_word_vector(w) for w in words])
        
        # 2. Calculate the L2 norm (magnitude) of each vector
        norms = np.linalg.norm(vector_matrix, axis=1, keepdims=True)
        
        # Prevent division by zero for Out-Of-Vocabulary (OOV) words
        norms[norms == 0] = 1 
        
        # 3. Normalize the matrix (divide each vector by its magnitude)
        normalized_matrix = vector_matrix / norms
        
        # 4. The dot product of a normalized matrix with its transpose 
        # instantly gives the pairwise cosine similarity for all combinations
        similarity_matrix = np.dot(normalized_matrix, normalized_matrix.T)
        
        # Clip to avoid floating point errors slightly outside [-1, 1]
        return np.clip(similarity_matrix, -1.0, 1.0)